# <h1 align="center"><b>Applio for Kaggle</b></h1>
<p align="center">A high-quality, lightweight voice conversion web interface and voice AI tool.</p>

<p align="center">
  <a href="https://discord.gg/wY7gmqTyEV" target="_blank">☎️ Discord Support</a> •
  <a href="https://github.com/IAHispano/Applio-app" target="_blank">💻 GitHub</a> •
  <a href="https://github.com/IAHispano/Applio-app/blob/main/TERMS_OF_USE.md" target="_blank">📜 Terms of Use</a>
</p>

---

> ⚠️ **IMPORTANT KAGGLE SETTINGS**:
> Before running the cells below, make sure to configure your notebook in the right sidebar:
> 1. **Settings ➔ Accelerator**: Choose **GPU T4 x2** or **GPU P100**.
> 2. **Settings ➔ Internet**: Toggle **Internet ON** (required to install packages, download RVC models, and generate the public web tunnel).

### **Step 1: Install Applio & Dependencies**
Clones the repository into `/kaggle/working/Applio`, installs audio and RVC prerequisites, and builds the web application.

In [ ]:
# @title Install Applio
import os
from pathlib import Path
from IPython.display import clear_output

REPO_URL = "https://github.com/IAHispano/Applio-app.git"
REPO_PATH = Path("/kaggle/working/Applio")

%cd /kaggle/working

if not REPO_PATH.exists():
    print("📥 Cloning Applio repository...")
    !git clone --depth 1 {REPO_URL} Applio
else:
    print("🔄 Applio directory exists, pulling updates...")
    %cd {REPO_PATH}
    !git pull

%cd {REPO_PATH}
clear_output()

print("📦 Installing system dependencies...")
!apt-get update -qq
!apt-get install -qq -y portaudio19-dev ffmpeg psmisc > /dev/null 2>&1

print("⚡ Installing Python dependencies with uv...")
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match --system
!uv pip install -q pyngrok --system

print("📥 Downloading core RVC prerequisites...")
!python core.py prerequisites --models --pretraineds-hifigan

print("🌐 Setting up Node.js & building Web Application...")
print("⬢ Installing Node.js 22 (required by pnpm 11 / Next.js 15)...")
!curl -fsSL https://deb.nodesource.com/setup_22.x | sudo -E bash - > /dev/null 2>&1
!sudo apt-get install -y nodejs > /dev/null 2>&1
!node --version
!npm install -g -q pnpm > /dev/null 2>&1
!pnpm install --frozen-lockfile > /dev/null 2>&1 || pnpm install > /dev/null 2>&1
!pnpm build

print("🔗 Installing Cloudflare & LocalTunnel utilities...")
!curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!npm install -g -q localtunnel > /dev/null 2>&1

clear_output()
print("✅ Installation complete! Run Step 2 below to launch Applio.")

### **Step 2: Start Applio Web UI**
Launches the Applio backend engine, Next.js web application, and tunneling service to provide a public URL.

In [ ]:
# @title Start Applio Web UI
tunnel_service = "Cloudflare (Recommended - Free, No Token)"  # @param ["Cloudflare (Recommended - Free, No Token)", "Ngrok (Requires Token)", "LocalTunnel (Free)"]
ngrok_authtoken = ""  # @param {type:"string"}
enable_tensorboard = False  # @param {type:"boolean"}

import os
import re
import time
import subprocess
import urllib.request
from IPython.display import display, HTML, clear_output

%cd /kaggle/working/Applio

REPO = os.getcwd()

def launch(cmd, log, extra_env=None):
    """Start a process detached from this cell (own session): it keeps running
    after the cell finishes AND after you stop/interrupt the cell. The stop
    button sends SIGINT to the cell's process group; start_new_session puts
    the child in a new session so the signal never reaches it. Only a runtime
    recycle (or the Stop cell below) ends it. Output goes to a file, never a
    pipe, so a full buffer can never block it."""
    env = dict(os.environ)
    env.update(extra_env or {})
    logf = open(log, "wb")
    return subprocess.Popen(
        cmd,
        stdin=subprocess.DEVNULL,
        stdout=logf,
        stderr=subprocess.STDOUT,
        start_new_session=True,
        cwd=REPO,
        env=env,
    )


def wait_for_url(log, pattern, timeout=30, group=0):
    t0 = time.time()
    while time.time() - t0 < timeout:
        time.sleep(1)
        try:
            with open(log, errors="replace") as f:
                m = re.search(pattern, f.read())
            if m:
                return m.group(group) if m.lastindex else m.group(0)
        except OSError:
            pass
    return None

# Terminate existing servers
!fuser -k 3000/tcp 2>/dev/null; fuser -k 8000/tcp 2>/dev/null; fuser -k 6006/tcp 2>/dev/null; true

if enable_tensorboard:
    %load_ext tensorboard
    %tensorboard --logdir logs --port 6006 --bind_all

print("🚀 Starting Applio backend engine (detached)...")
launch(["node", "app/api/dist/index.js"], "/tmp/applio-api.log", {"API_PORT": "8000"})

print("🌐 Starting Applio Next.js frontend (detached)...")
launch(
    ["node", "app/web/.next/standalone/server.js"],
    "/tmp/applio-web.log",
    {"PORT": "3000", "HOSTNAME": "0.0.0.0"},
)

# Wait for server readiness
print("⏳ Waiting for server readiness...")
ready = False
for _ in range(30):
    try:
        r = urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=1)
        if r.status == 200:
            ready = True
            break
    except Exception:
        time.sleep(1)

if not ready:
    print("⚠️ Warning: API health check timed out. Showing recent backend logs:")
    !tail -n 15 /tmp/applio-api.log
    !tail -n 15 /tmp/applio-web.log

print("🌐 Waiting for web frontend...")
front_ready = wait_for_url("/tmp/applio-web.log", r"Ready in", 30)
if not front_ready:
    print("⚠️ Warning: web frontend did not report ready. Last log lines:")
    !tail -n 15 /tmp/applio-web.log

clear_output()

def show_card(url, tunnel_name, extra_info=""):
    html = f"""
    <div style="font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
                background: #141414; border: 1px solid rgba(255, 255, 255, 0.12);
                border-radius: 14px; padding: 22px 26px; max-width: 540px; margin: 16px 0;
                box-shadow: 0 10px 30px rgba(0,0,0,0.5); color: #f5f5f5;">
        <div style="display: flex; align-items: center; justify-content: space-between; margin-bottom: 12px;">
            <div style="display: flex; align-items: center; gap: 8px;">
                <span style="font-size: 20px; font-weight: 700; letter-spacing: -0.02em;">Applio</span>
                <span style="font-size: 11px; font-weight: 500; color: #a3a3a3; background: rgba(255,255,255,0.08);
                             border: 1px solid rgba(255,255,255,0.1); border-radius: 4px; padding: 1px 6px;">Kaggle</span>
            </div>
            <span style="font-size: 12px; color: #4ade80; font-weight: 500;">● Online ({tunnel_name})</span>
        </div>
        <p style="font-size: 13px; color: #a3a3a3; margin: 0 0 16px 0; line-height: 1.4;">
            Your Applio web instance is ready. Click the button below to open the application in your browser:
        </p>
        <a href="{url}" target="_blank"
           style="display: inline-block; background: #ffffff; color: #0a0a0a; font-size: 13px; font-weight: 600;
                  padding: 10px 22px; border-radius: 999px; text-decoration: none; box-shadow: 0 2px 8px rgba(0,0,0,0.2);">
            Open Applio Web App ↗
        </a>
        <div style="margin-top: 14px; font-size: 12px; color: #737373;">
            Public Link: <a href="{url}" target="_blank" style="color: #60a5fa; word-break: break-all;">{url}</a>
        </div>
        {extra_info}
    </div>
    """
    display(HTML(html))

if "Cloudflare" in tunnel_service:
    os.system("pkill -f cloudflared 2>/dev/null")
    launch(["cloudflared", "tunnel", "--url", "http://127.0.0.1:3000"], "/tmp/applio-tunnel.log")
    tunnel_url = wait_for_url("/tmp/applio-tunnel.log", r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", 30)
    if tunnel_url:
        show_card(tunnel_url, "Cloudflare")
    else:
        print("❌ Could not obtain Cloudflare tunnel URL.")

elif "Ngrok" in tunnel_service:
    if not ngrok_authtoken or "http" in ngrok_authtoken or len(ngrok_authtoken.strip()) < 10:
        print("❌ Please enter your Ngrok Authtoken in the cell settings.")
        print("Obtain your free token here: https://dashboard.ngrok.com/get-started/your-authtoken")
    else:
        from pyngrok import ngrok as _ng
        _ng_bin = _ng.get_ngrok_path()
        os.system(f"pkill -f '{_ng_bin} http' 2>/dev/null")
        os.system(f"'{_ng_bin}' config add-authtoken {ngrok_authtoken.strip()} >/dev/null 2>&1")
        launch([_ng_bin, "http", "3000"], "/tmp/applio-tunnel.log")
        tunnel_url = wait_for_url("/tmp/applio-tunnel.log", r"Forwarding\s+(https://\S+)", 25, group=1)
        if tunnel_url:
            show_card(tunnel_url, "Ngrok")
        else:
            print("❌ Could not obtain Ngrok tunnel URL. Check the token and /tmp/applio-tunnel.log.")

elif "LocalTunnel" in tunnel_service:
    os.system("pkill -f 'lt --port' 2>/dev/null")
    endpoint_ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode("utf8").strip()
    launch(["lt", "--port", "3000"], "/tmp/applio-tunnel.log")
    tunnel_url = wait_for_url("/tmp/applio-tunnel.log", r"your url is:\s*(\S+)", 25, group=1)
    if tunnel_url:
        extra = f"""<div style="margin-top: 10px; font-size: 12px; color: #fbbf24;">
            ⚠️ <b>Tunnel Password IP:</b> <code style="background: rgba(255,255,255,0.1); padding: 2px 6px; border-radius: 4px;">{endpoint_ip}</code> (Enter this if prompted)
        </div>"""
        show_card(tunnel_url, "LocalTunnel", extra)
    else:
        print("❌ Could not obtain LocalTunnel URL.")

print("\n✅ Servers are detached: you can stop this cell, they keep running.")
print("Logs: /tmp/applio-api.log /tmp/applio-web.log /tmp/applio-tunnel.log")
print("To shut everything down, run the 'Stop Applio servers' cell below.")

In [ ]:
# @title Stop Applio servers
import os
os.system("fuser -k 3000/tcp 2>/dev/null; fuser -k 8000/tcp 2>/dev/null; fuser -k 6006/tcp 2>/dev/null; true")
os.system("pkill -f cloudflared 2>/dev/null; pkill -f 'lt --port' 2>/dev/null; pkill -f 'ngrok http' 2>/dev/null")
print("🛑 Applio servers stopped.")
